## ChatBot
We will be building a chatbot which will have the whole session history


In [1]:
#loading the API keys
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv('GROQ_API_KEY')


In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct",groq_api_key=groq_api_key)
model


/home/hp/Documents/GenAI/LangChain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7f75f5e2ad10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f75f5e2b370>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
#messages 
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Nithya Sai and I am an AI Engineer")])

AIMessage(content="Nice to meet you, Nithya Sai! I'm glad to hear that you're an AI Engineer. That's a fascinating field with so much potential for innovation and impact. What kind of AI projects are you currently working on? Are you focused on areas like natural language processing, computer vision, or perhaps reinforcement learning?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 25, 'total_tokens': 88, 'completion_time': 0.142646993, 'completion_tokens_details': None, 'prompt_time': 0.000151268, 'prompt_tokens_details': None, 'queue_time': 0.045215622, 'total_time': 0.142798261}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0bee-e7c4-7c53-bce0-6ae26b603056-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 63, 'total_token

In [4]:
from langchain_core.messages import AIMessage
ai_Message=AIMessage(content="Hello Nithya Sai! It's nice to meet you. As an AI Engineer, you must be working on some exciting projects. What kind of AI projects are you currently involved in? Are you focusing on areas like natural language processing, computer vision, or perhaps reinforcement learning?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 25, 'total_tokens': 81, 'completion_time': 0.128785699, 'completion_tokens_details': None, 'prompt_time': 0.000145038, 'prompt_tokens_details': None, 'queue_time': 0.044462651, 'total_time': 0.128930737}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0bca-da76-7d31-b389-ae8129f078c8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 56, 'total_tokens': 81})

model.invoke(
    [
        HumanMessage(content="Hi, My name is Nithya Sai and I am an AI Engineer"),
        ai_Message,
        HumanMessage(content="hey! what is my name and What do i do?")

    ]
    
)


AIMessage(content='Your name is Nithya Sai, and you are an AI Engineer!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 102, 'total_tokens': 118, 'completion_time': 0.036456655, 'completion_tokens_details': None, 'prompt_time': 0.00258041, 'prompt_tokens_details': None, 'queue_time': 0.0447043, 'total_time': 0.039037065}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0bee-e969-76d1-b473-de2c10f79119-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 102, 'output_tokens': 16, 'total_tokens': 118})

In [ ]:
## As we can see in the above chat ,the model is able to store my information and give the answer.
#This depends on the context window size of the model

## Message history
We can make our model stateful and store the chat history.

In [14]:
#to store the messages history in memory
from langchain_community.chat_message_histories import ChatMessageHistory
# template to define how to store the messages
from langchain_core.chat_history import BaseChatMessageHistory
# to identify the session and update the message history
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}
# it works without the return type also
def get_session_history(session_id:str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history=get_session_history)





In [15]:
config={"configurable":{"session_id":"Chat1"}}


In [16]:
#session1 
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Nithya Sai and I am an AI Engineer")],
    config=config
)

In [17]:
response.content

"Hello Nithya Sai! It's nice to meet you. As an AI Engineer, you must be working on some exciting projects. What kind of AI-related work are you currently involved in? Are you focusing on areas like natural language processing, computer vision, or perhaps reinforcement learning? I'd love to hear more about your experiences and interests!"

In [18]:
#lets use the history
response_name=with_message_history.invoke(
    [HumanMessage(content="What is my name")],
    config=config
)

In [19]:
#we are receiving the response with context given previously
# It is using the message history stored
response_name

AIMessage(content="Your name is Nithya Sai, and you're an AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 106, 'total_tokens': 121, 'completion_time': 0.037576239, 'completion_tokens_details': None, 'prompt_time': 0.002895776, 'prompt_tokens_details': None, 'queue_time': 0.044401664, 'total_time': 0.040472015}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0bf5-4cd2-70b2-8207-d23989fbf935-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 106, 'output_tokens': 15, 'total_tokens': 121})

In [21]:
#lets use the different session
config2={"configurable":{"session_id":"chat2"}}
response_chat2=with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config2
)
response_chat2.content

"I don't have access to personal information, so I don't know your name. I'm here to provide general information and answer questions, though! Is there something specific you'd like to know or discuss?"

In [22]:
## Prompt template -- to send messages in a structured way 
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
    ("system","You are a helpful assistant.Answer all the questions best of your knowledge"),
    MessagesPlaceholder(variable_name="messages")
    ]

)
chain= prompt| model

In [23]:
chain.invoke({
    "messages":[HumanMessage(content="Hi, My name is Nithya Sai")]
})

AIMessage(content='Hello Nithya Sai! Nice to meet you! How are you doing today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 38, 'total_tokens': 56, 'completion_time': 0.046302777, 'completion_tokens_details': None, 'prompt_time': 0.001391603, 'prompt_tokens_details': None, 'queue_time': 0.346324426, 'total_time': 0.04769438}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0bfe-c3e2-78e0-9392-d11338e0af10-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 18, 'total_tokens': 56})

In [26]:
with_message_prompt_history=RunnableWithMessageHistory(chain,get_session_history=get_session_history
                                                       )

In [32]:
config3={"configurable":{"session_id":"chat3_chain"}}
response_chat3=with_message_prompt_history.invoke(
    [HumanMessage(content="Hi, My name is Nithya Sai")],
    config=config3
)
response_chat3.content

"You've already told me that! Hi again, Nithya Sai! How's your day going so far?"

In [34]:
response=with_message_prompt_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config3
) 
response

AIMessage(content='Your name is Nithya Sai!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 121, 'total_tokens': 130, 'completion_time': 0.020656077, 'completion_tokens_details': None, 'prompt_time': 0.0030164, 'prompt_tokens_details': None, 'queue_time': 0.04519008, 'total_time': 0.023672477}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c05-177b-7d52-8686-c55fb9d0a991-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens': 9, 'total_tokens': 130})

In [51]:
## Lets use two parameters in the template

prompt_variables=ChatPromptTemplate.from_messages(
    [                                    
    ("system","You are a helpful assistant. Answer all the questions to best of your ability in {language}"),
    MessagesPlaceholder(variable_name="messages")
    ]
)

In [63]:
chain=prompt_variables| model

In [65]:
response=chain.invoke(
    {"messages":[HumanMessage(content="HI my name is Nithya Sai")],"language":"telugu"}

)
response

AIMessage(content='నమస్తే నిత్య సాయి! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 41, 'total_tokens': 60, 'completion_time': 0.043728219, 'completion_tokens_details': None, 'prompt_time': 0.000224508, 'prompt_tokens_details': None, 'queue_time': 0.045590012, 'total_time': 0.043952727}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c26-c7d0-7ac3-bac7-3941f4077f7e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 19, 'total_tokens': 60})

In [54]:
## Lets add the Chat History 
# now need to mention the messages variable if we use dict as input 
message_history_with_variables=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")
message_history_with_variables

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  messages: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x7f75f48081f0>, input_messages_key='messages', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [67]:
config={"configurable":{"session_id":"variable_chat1"}}
## keep the messages variable at last
response_final=message_history_with_variables.invoke(
    {"language":"telugu","messages":[HumanMessage(content="Give me your system instruction")]},
    config=config,
)
response_final.content

'నాకు ఇచ్చిన సిస్టమ్ సూచనలు:\n\n1. **వినియోగదారు భాషలో ప్రతిస్పందించడం**: వినియోగదారుడు ఒక నిర్దిష్ట భాషలో ఇన్పుట్ అందిస్తే, అదే భాషలో ప్రతిస్పందించడం.\n2. **వినియోగదారు అందించిన సమాచారాన్ని ఉపయోగించడం**: వినియోగదారుడు వారి పేరు లేదా ఇతర వ్యక్తిగత వివరాలను అందించినట్లయితే, ఆ సమాచారాన్ని ఉపయోగించి సంభాషణను వ్యక్తిగతీకరించడం.\n3. **సహాయక మరియు సమాచారమైన ప్రతిస్పందనలు అందించడం**: వినియోగదారుడు అడిగిన ప్రశ్నలకు ఖచ్చితమైన మరియు సహాయక ప్రతిస్పందనలు అందించడం మరియు ఉత్పాదక సంభాషణలలో పాల్గొనడం.\n4. **స్నేహపూర్వక మరియు గౌరవప్రదమైన స్వరాన్ని నిర్వహించడం**: వినియోగదారులతో స్నేహపూర్వక, సన్నిహిత మరియు గౌరవప్రదమైన పద్ధతిలో సంభాషించడం.\n5. **సంభాషణ ప్రవాహాన్ని అనుసరించడం**: సంభాషణలు తార్కికంగా ప్రవహించేలా మరియు వినియోగదారు ఇన్పుట్\u200cకు సరిపోయేలా ప్రతిస్పందించడం.\n\nఇప్పుడు, నిత్య సాయి, మీరు ఏమి మాట్లాడాలనుకుంటున్నారు?'

In [68]:
response=message_history_with_variables.invoke(
    {
        "messages":[("human","what is my name")],
        "language":"telugu"
    },
    config=config

)

In [69]:
response

AIMessage(content='నిత్య సాయి! మీ పేరు నిత్య సాయి అని నాకు గుర్తుంది!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 1608, 'total_tokens': 1636, 'completion_time': 0.070635124, 'completion_tokens_details': None, 'prompt_time': 0.046885288, 'prompt_tokens_details': None, 'queue_time': 0.076473471, 'total_time': 0.117520412}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c29-6827-7d30-8ddd-3d5e588f10d3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1608, 'output_tokens': 28, 'total_tokens': 1636})

In [70]:
store["variable_chat1"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi! My name is Nithya Sai', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Nithya Sai! It's nice to meet you! How are you doing today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 42, 'total_tokens': 61, 'completion_time': 0.043681251, 'completion_tokens_details': None, 'prompt_time': 0.000227107, 'prompt_tokens_details': None, 'queue_time': 0.045284453, 'total_time': 0.043908358}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c1b-77f0-7a92-85c8-852b53e57616-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 19, 'total_tokens': 61}), HumanMessage(content='Hi! My name is Nithya Sai', additional_kwargs={}, response_metadata={}), AIMessage(conte

## Managing the conversation history
If the conversation history is not managed then it grows beyond the context size and creates issues. 

In [93]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=30,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
    HumanMessage(content="I like Don movie")
    
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like Don movie', additional_kwargs={}, response_metadata={})]

In [81]:
## in above the Hi i am bob is gone

from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough
chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages"),language=itemgetter("language")| trimmer)
    | prompt_variables
    |model
)

In [84]:
response=chain.invoke(
    {
        "messages":messages+[HumanMessage(content="What ice cream I like?")],
        "language":"telugu"
    }
)
response

AIMessage(content='You mentioned earlier that you like vanilla ice cream!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 201, 'total_tokens': 212, 'completion_time': 0.025161234, 'completion_tokens_details': None, 'prompt_time': 0.006869887, 'prompt_tokens_details': None, 'queue_time': 0.136276793, 'total_time': 0.032031121}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c3c-d7f5-7cf0-8a85-5506af074ff6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 201, 'output_tokens': 11, 'total_tokens': 212})

In [99]:
##lets use the MessageHistory
with_message_history=RunnableWithMessageHistory(
    chain,get_session_history=get_session_history,input_messages_key="messages"
)
config={"configurable":{"session_id":"Chat6"}}


In [100]:
response=with_message_history.invoke(
    {
        "language":"telugu",
        "messages":messages+[HumanMessage(content="What is my name")]
    },
    config=config

)
response

AIMessage(content='Your name is Bob!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 208, 'total_tokens': 214, 'completion_time': 0.015513072, 'completion_tokens_details': None, 'prompt_time': 0.005840669, 'prompt_tokens_details': None, 'queue_time': 0.045044731, 'total_time': 0.021353741}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c42-acbe-7eb0-a699-cf400bf43dbf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 208, 'output_tokens': 6, 'total_tokens': 214})

In [101]:
response=with_message_history.invoke(
    {
        "language":"telugu",
        "messages":messages+[HumanMessage(content="What Ice cream do i like")]
    },
    config=config

)
response

AIMessage(content='You like vanilla ice cream!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 323, 'total_tokens': 330, 'completion_time': 0.016127235, 'completion_tokens_details': None, 'prompt_time': 0.008910906, 'prompt_tokens_details': None, 'queue_time': 0.045022374, 'total_time': 0.025038141}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c42-bf34-73e0-9850-2d7c64967811-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 323, 'output_tokens': 7, 'total_tokens': 330})

In [102]:
trimmer.invoke(messages)
store["Chat6"]

InMemoryChatMessageHistory(messages=[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}), HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}), AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}), AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}), AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}), AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}), AIMessage(content